In [ ]:
# This test program aims to output a list of n_mc coordinates within a sample.
# This program will use ray-tracing and the odd-even rule to determine whether a random coordinate is within a sample.

using MeshIO
using FileIO
using BenchmarkTools
using StaticArrays
using GeometryBasics
include("../../src/sampling.jl")
using .sampling

In [2]:
# Setting the desired number of MC sample points.

const n_mc = 10

10

In [ ]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.

# Dummy multiple-crystal sample comprising 7 icospheres with 80 faces each.
stl = load("../Test_STLs/7_Icospheres80.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

560

In [4]:
# Storing the coordinates into a vector of length n_mc.

mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)
len_i = Vector{Float32}(undef, n_mc)

10-element Vector{Float32}:
   1.381571f10
   7.72f-43
   0.0
   0.0
   1.3815726f10
   7.72f-43
   0.0
   0.0
  -0.18221067
 NaN

In [5]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = sampling.ve_calc(vertices, indices)

(SVector{3, Float32}[[-0.58778524, 0.809017, 0.0], [0.0, -1.0, 0.0], [0.58778524, 0.809017, 0.0], [-0.95105654, 0.309017, 0.0], [0.58778524, -0.809017, 0.0], [-0.4253254, -0.309017, 0.8506508], [0.4253254, 0.309017, -0.8506508], [-0.68819094, 0.5, -0.5257311], [-0.5257311, 0.0, -0.8506508], [-0.68819094, -0.5, -0.5257311]  …  [-0.4253254, -0.309017, -2.1493492], [0.16245985, -0.5, -2.1493492], [0.68819094, -0.5, -2.474269], [0.16245985, -0.5, -2.1493492], [0.5257311, 0.0, -2.1493492], [0.68819094, 0.5, -2.474269], [0.5257311, 0.0, -2.1493492], [0.16245985, 0.5, -2.1493492], [-0.26286554, 0.809017, -2.474269], [0.16245985, 0.5, -2.1493492]], SVector{3, Float32}[[0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [0.10040575, -0.309017, 0.5257311], [0.10040569, 0.309017, 0.5257311], [-0.4253254, 0.309017, -0.3249197], [-0.16245985, 0.5, 0.3249197], [0.10040569, 0.309017, 0.5257311], [0.36327124, 0.5, 0.0], [0.5257311, 0.0, -0.3249197]  …  [0.16245985,

In [6]:
# Generating the sample points (and initial path lengths).

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = sampling.aabb_3d(vertices)
sampling.sample!(ranges, e2s, e3s, v1s, len_i, mc_coords, true, n_faces, n_mc)
display(mc_coords)

10-element Vector{SVector{3, Float32}}:
 [0.466334, 0.16029346, -2.3053179]
 [3.3918548, -0.56817967, -0.45306286]
 [-0.24906921, 0.55770767, -2.5539398]
 [-3.1484797, 0.32620814, -0.14042504]
 [-2.8382752, 0.8456823, -0.027456863]
 [-0.21973988, 3.1574695, 0.78202707]
 [-0.78750443, -0.028361619, 3.4523365]
 [0.031068722, -0.12789036, -0.20452893]
 [0.11046453, -2.3065932, -0.34535643]
 [-0.21071488, 0.3805394, 0.8165374]

In [8]:
# Benchmarking this sample point generation function.

@benchmark sampling.sample!(ranges, e2s, e3s, v1s, len_i, mc_coords, true, n_faces, n_mc)

BenchmarkTools.Trial: 3629 samples with 1 evaluation per sample.
 Range (min … max):  374.300 μs …  19.755 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       1.216 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.353 ms ± 689.705 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

         ▂▅▆▆▄█▆▄▇▃▄▃▁▂▁                                         
  ▁▂▃▅▆▇███████████████████▆▆▆▄▅▄▄▃▄▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁ ▄
  374 μs           Histogram: frequency by time          3.6 ms <

 Memory estimate: 9.08 KiB, allocs estimate: 10.